### Computational Guided Inquiry for Modeling Earth's Climate (Neshyba & Eklof, 2026)

# Cambio 4.0
In previous versions of Cambio, the atmosphere-to-land carbon flux was calculated as $F_{al} = k_{al0} +  k_{al1} \times C_{atm}$, in which $k_{al1}$ was described as a measure of the strength of Earth's natural $CO_2$ fertilization capability. Recently, Ke et al, 2024 (https://arxiv.org/abs/2407.12447) have documented reduction in this strength -- an **impairment in $CO_2$ fertilization**. The implementation of this impairment in Cambio 4.0 consists of the addition of a sigmoid term to the atmosphere-to-land flux, Eq. 2:
$$
F_{la} =  k_{la} \ \ \ (1) 
$$

$$
F_{al} = k_{al0} +  k_{al1} \times C_{atm} \times  \sigma_{floor}(T_{anomaly})  \ \ \ (2)$$

$$
F_{oa} = k_{oa} \times (1+DC\times T_{anomaly}) C_{ocean} \ \ \ (3)
$$

$$
F_{ao} = k_{ao} C_{atm} \ \ \ (4)
$$

$$
F_{ha} = \epsilon(t) \ \ \ (5)
$$

Parameters of the sigmoid function introduced in Eq. 2 ($T_{anomaly}^*$, $\Delta T$, and $\sigma_{floor}$) are determined by matching to the observations of Ke et al.

## Uploading your climate emissions scenario
As before, you'll need to upload a climate emissions scenario file to the current folder. 

## Learning goals
1. I can describe how to implement $CO_2$ fertilization impairment in an Euler loop.
3. I can describe how terrestrial sequestration impairment affects key climate predictions, including the magnitude and timing of anthropogenic maximum warming.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt; plt.rc("figure", figsize=(12,8))
import meclib.cl as cl
from copy import copy as makeacopy

### Loading your favorite scheduled flow
In the cell below, load in your scheduled flows file. It'll be most convenient if you use the following naming convention: 

    time, eps, epsdictionary_from_file = cl.LoadMyScenario('...')

(but of course supplying the name of your own scheduled flows file). 

In [ ]:
# Your code here


### Creating a dictionary for climate parameters
In the cell below, we use the CreateClimateParams function to create a dictionary of climate parameters.

In [ ]:
ClimateParams = cl.CreateClimateParams(epsdictionary_from_file)
display(ClimateParams)

### Getting a sense of the impact of the $CO_2$ fertilization impairment
The cell below provides a sense of the potential impact of the the $CO_2$ fertilization impairment due to rising temperatures, by plotting $F_{atm->land} = k_{al0} +  k_{al1} \times \sigma_{floor}(T_{anomaly}) \times C_{atm}$ as a function of the temperature anomaly. Parameters controlling the sigmoid function ($T_{anomaly}^*$, $\Delta T$, and $\sigma_{floor}$) were determined based on a fit to data given in Ke et al, 2024 (see the papers by Neshyba et al, https://journals.ametsoc.org/view/journals/bams/107/1/BAMS-D-25-0069.1.xml, and by Ke et al, https://arxiv.org/abs/2407.12447).

(We should emphasize that a proper assessment of $CO_2$ fertilization impairment should be made from an integrated simulation, in which the driver is included as part of the solution (Euler loop) of Eqs. 1-5 in the introduction. You'll do that a little farther down in this Notebook.)

In [ ]:
# A range of atmospheric CO2 amounts (preindustrial to 2x that)
C_atm_array = np.linspace(590,2*590)
C_atm_array_ppm = C_atm_array/2.12

# Get the temperature anomaly that would result from that, assuming no ice-albedo feedback
T_anomaly = cl.Diagnose_T_anomaly(C_atm_array, 0.3, ClimateParams)

# Other parameters we'll need for this comparison 
k_al0 = ClimateParams['k_al0']
k_al1 = ClimateParams['k_al1']
k_al1_Tstar = ClimateParams['k_al1_Tstar']
k_al1_deltaT = ClimateParams['k_al1_deltaT']
fractional_k_al1_floor = ClimateParams['fractional_k_al1_floor']

# Terrestrial atmosphere-to-land flux with and without terrestrial sequestration feedback
F_al_without_tsf = k_al0 + k_al1*C_atm_array
F_al_with_tsf_extrap = k_al0 + k_al1*C_atm_array*fractional_k_al1_floor
F_al_with_tsf = k_al0 + k_al1*C_atm_array*cl.sigmafloor(T_anomaly,k_al1_Tstar,k_al1_deltaT,fractional_k_al1_floor)

# Plot them side by side
iextrap = int(len(C_atm_array_ppm)/2)
plt.figure()
plt.plot(C_atm_array_ppm,F_al_without_tsf,'k',label="No TSF")
plt.plot(C_atm_array_ppm,F_al_with_tsf,'r',label="With TSF (tipping point $T^*_{anomaly}$="+str(k_al1_Tstar)+"$^oC$)")
plt.plot(C_atm_array_ppm[iextrap:],F_al_with_tsf_extrap[iextrap:],'r.')
plt.xlabel('Carbon in atmosphere (ppm)')
plt.ylabel('F_al (GtC/yr)')
plt.title('Est. impact of terrestrial sequestration feedback on atm-to-land flux')
plt.legend()
plt.grid()

### Pause for analysis
For both questions, use the graph we just made to estimate your answer. Remember that *impairment* means the difference between these two curves. Answers to both questions should be in GtC/yr.

1. Using the current concentration of $CO_2$ in the atmosphere, how much has $CO_2$ fertilization already been impaired so far?
1. How much impairment can we expect if atmospheric $CO_2$ were to rise to double its pre-industrial amount ($280 \times 2 = 560 \ ppm$)? 

YOUR ANSWER HERE

### Duplicating Cambio3.0
Below, we reproduce Cambio3.0.

In [ ]:
def PropagateCS_Cambio3(previousClimateState, ClimateParams, dt, F_ha):
    """Propagates the state of the climate, with a specified anthropogenic carbon flux"""
    """Returns a new climate state"""

    # Extract constants from ClimateParams
    k_la = ClimateParams['k_la']
    k_al0 = ClimateParams['k_al0']
    k_al1 = ClimateParams['k_al1']
    k_oa = ClimateParams['k_oa']
    k_ao = ClimateParams['k_ao']
    DC = ClimateParams['DC']
    preindustrial_albedo = ClimateParams['preindustrial albedo']
    fractional_albedo_floor = ClimateParams['fractional_albedo_floor']
    albedo_Tstar = ClimateParams['albedo_Tstar']
    albedo_delta_T = ClimateParams['albedo_delta_T']
    k_al1_Tstar = ClimateParams['k_al1_Tstar']
    k_al1_deltaT = ClimateParams['k_al1_deltaT']
    fractional_k_al1_floor = ClimateParams['fractional_k_al1_floor']
    
    # Extract concentrations and time from the previous climate state
    C_atm = previousClimateState['C_atm']
    C_ocean = previousClimateState['C_ocean']
    time = previousClimateState['time']
    
    # Extract the temperature from the previous climate state
    T_anomaly = previousClimateState['T_anomaly']
    
    # Calculate the albedo implied by that temperature anomaly
    albedo = cl.Diagnose_albedo(T_anomaly, ClimateParams)

    # Calculate a new temperature anomaly from that albedo
    T_anomaly = cl.Diagnose_T_anomaly(C_atm, albedo, ClimateParams)

    # Other diagnostics
    actual_temperature = cl.Diagnose_actual_temperature(T_anomaly)
    OceanSurfacepH = cl.Diagnose_OceanSurfacepH(C_atm,ClimateParams)
    
    # Get new fluxes (including the effect of temperature anomaly on the ocean-to-atmosphere flux)
    F_la = k_la    
    F_al = k_al0 + k_al1*C_atm
    F_oa = k_oa*C_ocean*(1+DC*T_anomaly)    
    F_ao = k_ao*C_atm

    # Get new concentrations of carbon that depend on the fluxes
    C_atm += (F_la + F_oa - F_ao - F_al + F_ha)*dt
    C_ocean += (F_ao - F_oa)*dt
    time += dt
    
    # Create a new climate state with these updates
    ClimateState = makeacopy(previousClimateState)
    ClimateState['C_atm'] = C_atm
    ClimateState['C_ocean'] = C_ocean
    ClimateState['F_al'] = F_al
    ClimateState['F_la'] = F_la
    ClimateState['F_ao'] = F_ao
    ClimateState['F_oa'] = F_oa
    ClimateState['F_ha'] = F_ha
    ClimateState['F_ocean_net'] = F_oa-F_ao
    ClimateState['F_land_net'] = F_la-F_al
    ClimateState['time'] = time
    ClimateState['T_anomaly'] = T_anomaly
    ClimateState['actual temperature'] = actual_temperature
    ClimateState['OceanSurfacepH'] = OceanSurfacepH
    ClimateState['albedo'] = albedo

    # Return the new climate state
    return ClimateState

# Run Cambio3.0
CS_Cambio3_list = cl.run_Cambio(PropagateCS_Cambio3, ClimateParams, time, eps)

# Choose items to plot
items_to_plot = [['C_atm','C_ocean'],['F_ha','F_ocean_net','F_land_net'],'T_anomaly','albedo','OceanSurfacepH']

# Plot those items
cl.CS_list_plots(CS_Cambio3_list,'Cambio3',items_to_plot)

### Cambio 4.0
Below, the goal is to enhance Cambio3.0 with the impairment of $CO_2$ fertilization implemented as in Eq. 2 of the introduction to this Notebook. Then make all the plots done above for Cambio3.0.

In [ ]:
# Your code here


### Pause for analysis
How does Cambio 4.0 compare to Cambio 3.0? Qualitatively the same, but different quantitatively? Or qualitatively different? It will be helpful to refer to the max/min printouts as you do so.

YOUR ANSWER HERE

### Validating and finishing up
Assuming all this has gone smoothly, don't forget to do a Kernel/Restart & Run All, run the whole notebook, and make sure there aren't any errors.